# Week 5 - Optimization Models for Picking Loans

This notebook carries out the following steps
  1. Read the saved pickle data and prepares it for analysis and load the classification models for default probability and regression models for returns
  2. Uses them to implement the baseline strategies from last week
  3. Uses the predicted default probabilities and returns in an optimization model to pick a set of loans to invest in (You can compare the optimization based portfolios to the strategies from last week)
  4. Performs a sensitivity analysis of the performance of the loan portfolio by re-solving the optimization models for various levels of risk

Things for you to do
- Change the number of clusters based on your assessment.
- Use the best model for predicting the returns of your chosen method from the last module and replace the regressionl model here with that.
- You can output any additional fields about the test loans you may want to examine later if they will help you explain your final loans picked by your optimization model.

Prepare your presentation. Your presentation should contain about at most 6 slides. Add an extra slide at the end if you tried the BONUS.
1. Begin by stating the objective of the presentation and what your objective is. Which questions do you seek to answer? What are the main points of the presentation?
2. (& 3) Describe the final portfolio that you obtained. What variable did you optimize for? Which constraints did you impose? Which method from the last module did you use to predict returns? What are the grades of the selected loans?
4. Compare the portfolio that you obtained with the one obtained in module 4. Which metrics are you using to compare them? You may want to consider factors such as risk, probability of default, returns and scalability.
5. State your conclusions. What is the main idea you wish to convey with the presentation? Based on the last two models, how should Jasmine invest her money? 

BONUS
- Do you think it is better to use the loan risk as we did in this script or to use the probability of default from Module 3? Implement a new optimization model using probability of default. Clearly explain your model and discuss its pros and cons. How does this new method compare to the one you obtained in this script? 
- Split your data set between short term and long-term loans. Re-compute this script for the two types of loans. Do you get better results?

In [ ]:
#conda config

In [ ]:
#conda install gurobi

In [1]:
# Load general utilities
# ----------------------
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.axes as ax
import datetime
import numpy as np
import pickle
import time
import seaborn as sns

# Load sklearn utilities
# ----------------------
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve, brier_score_loss, mean_squared_error, r2_score

from sklearn.calibration import calibration_curve

# Load classifiers
# ----------------
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import BaggingClassifier

# Other Packages
# --------------
from scipy.stats import kendalltau
from sklearn.neural_network import MLPRegressor
from sklearn import linear_model
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
## from gurobipy import *
from six import StringIO  
from IPython.display import Image  
from sklearn.tree import export_graphviz
import pydotplus
#from scipy.interpolate import spline

# Load debugger, if required
#import pixiedust
pd.options.mode.chained_assignment = None #'warn'

# suppress all warnings
import warnings
warnings.filterwarnings("ignore")

## 5.1 Load saved models.
Load the dictionary first, assign value when we need to.

In [2]:
# read saved models
infile = open('/Users/hrishikeshharishkumar/Documents/CMU Tepper - MSBA/Mini 1 - Fall 2023/Business Value through Integrative Analytics/Update 4 - Prescriptive Modeling - Ranking Using Predictors/saved models/week4_saved_models','rb')
saved_models = pickle.load(infile)
infile.close()
print('models loaded:', saved_models.keys())

# save into a new dict if there is anything new to save
models_to_save = saved_models.copy()

models loaded: dict_keys(['rf1213', 'reg_rf1213', 'reg_rf_separate1213', 'reg_mlp1213', 'l1_logistic1213'])


## 5.2 Build and test baseline investment strategies
Now we test several investment strategies using the learning models above

In [3]:
default_seed = 1
output_file = "output_sample"

In [4]:
# Create a function to print a line to our output file
def dump_to_output(key, value):
    with open(output_file, "a") as f:
        f.write(",".join([str(default_seed), key, str(value)]) + "\n")

In [5]:
def test_investments(data_dict,
                        classifier = None,
                        regressor = None,
                        strategy = 'Random',
                        num_loans = 1000,
                        random_state = default_seed,
                        output_to_file = True):
    '''
    This function tests a variety of investment methodologies and their returns.
    It will run its tests on the loans defined by the test_set element of the data
    dictionary.

    It is currently able to test four strategies
      - random: invest in a random set of loans
      - ranking: score each loan by probability of default, and only invest
                 in the "safest" loans (i.e., those with the lowest probabilities
                 of default)
      - regression: train a single regression model to predict the expected return
                    of loans in the past. Then, for loans we could invest in, simply
                    rank them by their expected returns and invest in that order.
      - two-stage: train two regression models to predict the expected return of
                   defaulted loans and non-defaulted loans in the training set. Then,
                   for each potential loan we could invest in, predict the probability
                   the loan will default, its return if it doesn't default and its
                   return if it does. Then, calculate a weighted combination of
                   the latter using the former to find a predicted return. Rank the
                   loans by this expected return, and invest in that order

    It expects the following parameters
      - data_dict: the dictionary containing both training and testing data;
                   returned by the prepare_data function
      - classifier: a fitted model object which is returned by the fit_classification function.
      - regressor: a fitted model object which is returned by the fit_regression function.
      - strategy: the name of the strategy; one of the three listed above
      - num_loans: the number of loans to be included in the test portfolio
      - num_samples: the number of random samples used to compute average return ()
      - random_state: the random seed to use when selecting a subset of rows
      - output_to_file: if the results will be saved to the output file

    The function returns a dictionary FOR EACH RETURN DEFINITION with the following entries
      - strategy: the name of the strategy
      - average return: the return of the strategy based on the testing set
      - test data: the updated Dataframe of testing data. Useful in the optimization section
    '''

    np.random.seed(random_state)

    # Retrieve the rows that were used to train and test  the
    # classification model
    train_set = data_dict['train_set']
    test_set = data_dict['test_set']

    #col_list = ['ret_PESS', 'ret_OPT', 'ret_INTa', 'ret_INTb']
    col_list = ['ret_OPT']

    # Create a dataframe for testing, including the score
    data_test = data.loc[test_set,:]
    out = {}

    for ret_col in col_list:

        if strategy == 'Random':
            # Randomize the order of the rows in the datframe
            data_test = data_test.sample(frac = 1).reset_index(drop = True)

            # Select num_loans to invest in
            pf_test = data_test[['funded_amnt',ret_col]].iloc[:num_loans]

            # Find the average return for these loans
            ret_test = np.dot(pf_test[ret_col],pf_test.funded_amnt)/np.sum(pf_test.funded_amnt)

            # Return
            out[ret_col] = {'strategy':strategy, 'average return':ret_test}

            # Dump the strategy performance to file
            if output_to_file:
                dump_to_output(strategy + "," + ret_col + "::average return", ret_test )

            continue

        elif strategy == 'Regression':

            colname = 'predicted_return_' + ret_col

            data_test[colname] = regressor[ret_col]['predicted_return']

            # Sort the loans by predicted return
            data_test = data_test.sort_values(by=colname, ascending = False).reset_index(drop = True)

            # Pick num_loans loans
            pf_test = data_test[['funded_amnt',ret_col]].iloc[:num_loans]

            # Find their return
            ret_test = np.dot(pf_test[ret_col],pf_test.funded_amnt)/np.sum(pf_test.funded_amnt)

            # Return
            out[ret_col] = {'strategy':strategy, 'average return':ret_test, 'test data':data_test}

            # Dump the strategy performance to file
            if output_to_file:
                dump_to_output(strategy + "," + ret_col + "::average return", ret_test )

            continue

        # Get the predicted scores, if the strategy is not Random or just Regression
        try:
            y_pred_score = classifier['y_pred_probs']
        except:
            y_pred_score = classifier['y_pred_score']

        data_test['score'] = y_pred_score


        if strategy == 'Ranking':
            # Sort the test data by the score
            data_test = data_test.sort_values(by='score').reset_index(drop = True)

            # Select num_loans to invest in
            pf_test = data_test[['funded_amnt',ret_col]].iloc[:num_loans]

            # Find the average return for these loans
            ret_test = np.dot(pf_test[ret_col],pf_test.funded_amnt)/np.sum(pf_test.funded_amnt)

            # Return
            out[ret_col] = {'strategy':strategy, 'average return':ret_test}

            # Dump the strategy performance to file
            if output_to_file:
                dump_to_output(strategy + "," + ret_col + "::average return", ret_test )

            continue


        elif strategy == 'Two-stage':

            # Load the predicted returns
            data_test['predicted_regular_return'] = regressor[ret_col]['predicted_regular_return']
            data_test['predicted_default_return'] = regressor[ret_col]['predicted_default_return']

            # Compute expectation
            colname = 'predicted_return_' + ret_col

            data_test[colname] = ( (1-data_test.score)*data_test.predicted_regular_return +
                                             data_test.score*data_test.predicted_default_return )

            # Sort the loans by predicted return
            data_test = data_test.sort_values(by=colname, ascending = False).reset_index(drop = True)

            # Pick num_loans loans
            pf_test = data_test[['funded_amnt',ret_col]].iloc[:num_loans]

            # Find their return
            ret_test = np.dot(pf_test[ret_col],pf_test.funded_amnt)/np.sum(pf_test.funded_amnt)

            # Return
            out[ret_col] = {'strategy':strategy, 'average return':ret_test, 'test data':data_test}

            # Dump the strategy performance to file
            if output_to_file:
                dump_to_output(strategy + "," + ret_col + "::average return", ret_test )

            continue

        elif strategy == 'Crystal-ball':

            # Sort the loans by realized return
            data_test = data_test.sort_values(by=ret_col, ascending = False).reset_index(drop = True)

            # Pick num_loans loans
            pf_test = data_test[['funded_amnt',ret_col]].iloc[:num_loans]

            # Find their return
            ret_test = np.dot(pf_test[ret_col],pf_test.funded_amnt)/np.sum(pf_test.funded_amnt)

            # Return
            out[ret_col] = {'strategy':strategy, 'average return':ret_test}

            # Dump the strategy performance to file
            if output_to_file:
                dump_to_output(strategy + "," + ret_col + "::average return", ret_test )

            continue

        else:
            return 'Not a valid strategy'

    return out

In [6]:
# 1. Load the data and engineer the features
## 1.1 Load data

# Read the data and features from the pickle
data, discrete_features, continuous_features, ret_cols = pickle.load( open( "/Users/hrishikeshharishkumar/CMU Mini 1 - Fall 2023 - Business Value through Integrative Analytics/Update 2: Diagnostic and Descriptive Analysis/PickleData/ret_data_Sep28.pickle", "rb" ) )

# Create the outcome
data["outcome"] = data.loan_status.isin(["Charged Off", "Default"])

# Create a feature for the length of a person's credit history at the
# time the loan is issued
data['cr_hist'] = (data.issue_d - data.earliest_cr_line) / np.timedelta64(1, 'M')
continuous_features.append('cr_hist')

# Randomly assign each row to a training and test set. We do this now
# because we will be fitting a variety of models on various time periods,
# and we would like every period to use the *same* training/test split
np.random.seed(default_seed)
data['train'] = np.random.choice([True, False], size = len(data), p = [0.7, 0.3])

# Create a matrix of features and outcomes, with dummies. Record the
# names of the dummies for later use
X_continuous = data[continuous_features].values

X_discrete = pd.get_dummies(data[discrete_features], dummy_na = True, prefix_sep = "::", drop_first = True)
discrete_features_dummies = X_discrete.columns.tolist()
X_discrete = X_discrete.values

X = np.concatenate( (X_continuous, X_discrete), axis = 1 )

y = data.outcome.values

train = data.train.values

## 1.2 Prepare functions to fit and evaluate models

def prepare_data(data_subset = np.array([True]*len(data)),
                    n_samples_train = 20000,
                    n_samples_test = 10000,
                    feature_subset = None,
                    date_range_train = (data.issue_d.min(), data.issue_d.max()),
                    date_range_test = (data.issue_d.min(), data.issue_d.max()),
                    random_state = default_seed):
    '''
    This function will prepare the data for classification or regression.
    It expects the following parameters:
      - data_subset: a numpy array with as many entries as rows in the
                     dataset. Each entry should be True if that row
                     should be used, or False if it should be ignored
      - n_samples_train: the total number of samples to be used for training.
                         Will trigger an error if this number is larger than
                         the number of rows available after all filters have
                         been applied
      - n_samples_test: as above for testing
      - feature_subect: A list containing the names of the features to be
                        used in the model. In None, all features in X are
                        used
      - date_range_train: a tuple containing two dates. All rows with loans
                          issued outside of these two dates will be ignored in
                          training
      - date_range_test: as above for testing
      - random_state: the random seed to use when selecting a subset of rows
      
    Note that this function assumes the data has a "Train" column, and will
    select all training rows from the rows with "True" in that column, and all
    the testing rows from those with a "False" in that column.
    
    This function returns a dictionary with the following entries
      - X_train: the matrix of training data
      - y_train: the array of training labels
      - train_set: a Boolean vector with as many entries as rows in the data
                  that denotes the rows that were used in the train set
      - X_test: the matrix of testing data
      - y_test: the array of testing labels
      - test_set: a Boolean vector with as many entries as rows in the data
                  that denotes the rows that were used in the test set
    '''
    
    np.random.seed(random_state)
        
    # Filter down the data to the required date range, and downsample
    # as required
    filter_train = ( train & (data.issue_d >= date_range_train[0]) &
                            (data.issue_d <= date_range_train[1]) & data_subset ).values
    filter_test = ( (train == False) & (data.issue_d >= date_range_test[0])
                            & (data.issue_d <= date_range_test[1]) & data_subset ).values
    
    filter_train[ np.random.choice( np.where(filter_train)[0], size = filter_train.sum() 
                                   - n_samples_train, replace = False ) ] = False
    filter_test[ np.random.choice( np.where(filter_test)[0], size = filter_test.sum() 
                                   - n_samples_test, replace = False ) ] = False
    
    # Prepare the training and test set
    X_train = X[ filter_train , :]
    X_test = X[ filter_test, :]
    if feature_subset != None:
        cols = [i for i, j in enumerate(continuous_features + discrete_features_dummies)
                                                     if j.split("::")[0] in feature_subset]
        X_train = X_train[ : , cols ]
        X_test = X_test[ : , cols ]
        
    y_train = y[ filter_train ]
    y_test = y[ filter_test ]
    
    # Scale the variables
    scaler = preprocessing.MinMaxScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # return training and testing data
    out = {'X_train':X_train, 'y_train':y_train, 'train_set':filter_train, 
           'X_test':X_test, 'y_test':y_test, 'test_set':filter_test}
    
    return out

## Implement the test strategies on 2012-13 data

In [35]:
## read saved models
reg_rf1213 = saved_models['reg_rf1213']
rf1213 = saved_models['rf1213']

In [36]:
# Read the data and features from the pickle
data, discrete_features, continuous_features, ret_cols = pickle.load( open( "/Users/hrishikeshharishkumar/CMU Mini 1 - Fall 2023 - Business Value through Integrative Analytics/Update 2: Diagnostic and Descriptive Analysis/PickleData/ret_data_Sep28.pickle", "rb" ) )

In [37]:
discrete_features.remove('verification_status')

In [38]:
discrete_features.remove('home_ownership')

In [39]:
discrete_features.remove('purpose')

In [40]:
continuous_features.remove('pct_tl_nvr_dlq')

In [41]:
continuous_features.remove('total_acc')

In [42]:
continuous_features.remove('tot_coll_amt') 

In [43]:
continuous_features.remove('funded_amnt_inv') 

In [44]:
data = data.drop(['verification_status','home_ownership','purpose','out_prncp','pct_tl_nvr_dlq','total_acc','acc_now_delinq','tot_coll_amt'], axis=1, errors='ignore')

In [45]:
data = data.drop(['earliest_cr_line','total_pymnt','recoveries','funded_amnt_inv','loan_status','last_pymnt_d','default'], axis=1, errors='ignore')

In [46]:
discrete_features

['grade', 'term', 'emp_length']

In [47]:
continuous_features

['loan_amnt',
 'funded_amnt',
 'installment',
 'annual_inc',
 'dti',
 'revol_bal',
 'delinq_2yrs',
 'open_acc',
 'pub_rec',
 'tot_cur_bal',
 'bc_util',
 'inq_last_6mths',
 'int_rate',
 'revol_util']

In [48]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 205608 entries, 0 to 235628
Data columns (total 25 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   loan_amnt       205608 non-null  float64
 1   funded_amnt     205608 non-null  float64
 2   term            205608 non-null  object 
 3   int_rate        205608 non-null  float64
 4   installment     205608 non-null  float64
 5   grade           205608 non-null  object 
 6   emp_length      195079 non-null  object 
 7   annual_inc      205608 non-null  float64
 8   issue_d         205608 non-null  object 
 9   dti             205608 non-null  float64
 10  delinq_2yrs     205608 non-null  float64
 11  open_acc        205608 non-null  float64
 12  pub_rec         205608 non-null  float64
 13  revol_bal       205608 non-null  float64
 14  revol_util      205608 non-null  float64
 15  tot_cur_bal     205608 non-null  float64
 16  bc_util         205608 non-null  float64
 17  inq_last_6

In [49]:
final_features = [i for i in discrete_features + continuous_features]
data_dict = prepare_data(feature_subset = final_features)

all_features = pd.Series(continuous_features + discrete_features_dummies)
idx = [i for i, j in enumerate(continuous_features + discrete_features_dummies)
                                                     if j.split("::")[0] in final_features]
selected_features = all_features[idx]
selected_features.reset_index(drop=True,inplace=True)

### 5.2.1 Random

In [57]:
#col_list = ['ret_PESS', 'ret_OPT', 'ret_INTa', 'ret_INTb']
col_list = ['ret_OPT']
test_strategy = 'Random'

print('strategy:',test_strategy)   
strat_random = test_investments(data_dict,strategy = test_strategy, 
                            num_loans = 20, output_to_file = False, random_state = 1)
for ret_col in col_list:
    print(ret_col + ': ' + str(strat_random[ret_col]['average return']))

strategy: Random
ret_OPT: 0.07562399137969766


### 5.2.2 Ranking

In [58]:
test_strategy = 'Ranking'

print('strategy:',test_strategy)
strat_rank = test_investments(data_dict, classifier=rf1213, strategy = test_strategy, 
                        num_loans = 20, output_to_file = False)

for ret_col in col_list:
    print(ret_col + ': ' + str(strat_rank[ret_col]['average return']))

strategy: Ranking
ret_OPT: 0.05307080321035306


### 5.2.3 Regression

In [59]:
test_strategy = 'Regression'

print('strategy:',test_strategy)
strat_reg = test_investments(data_dict, regressor=reg_rf1213, strategy = test_strategy, 
                        num_loans = 20)
for ret_col in col_list:
    print(ret_col + ': ' + str(strat_reg[ret_col]['average return']))

strategy: Regression
ret_OPT: 0.07284314472699509


### 5.2.4 Two-stage
#### Compute random forest regression:

In [60]:
## load model
reg_rf_separate1213 = saved_models['reg_rf_separate1213']

#### test two-stage stratgy

In [63]:
l1_logistic1213 = saved_models['l1_logistic1213']

In [64]:
test_strategy = 'Two-stage'


print('strategy:',test_strategy)
two_stage = test_investments(data_dict, classifier = l1_logistic1213, regressor = reg_rf_separate1213, 
                             strategy = test_strategy, num_loans = 20)

for ret_col in col_list:
    print(ret_col + ': ' + str(two_stage[ret_col]['average return']))

strategy: Two-stage
ret_OPT: 0.07636823827822127


## 5.3 Optimization
 In this section, we implement three different optimization models. To illustrate and compare these models we will only use the M1-PESS definition and the predicted returns from the previously tested two-stage strategy.

## Three optimization models to picks loans

### 5.3.1 Directly maximize total profit


In [55]:
#pip install gurobipy

In [56]:
import gurobipy as gp
from gurobipy import GRB

ret_col = 'ret_OPT'
test_pool = two_stage[ret_col]['test data']
num_var = test_pool.shape[0]
num_loans = 10

## first define cost vector
c = np.zeros(num_var) # cost vector
for i in range(num_var):
    c[i] = test_pool['predicted_return_'+ret_col].iloc[i]*test_pool.funded_amnt.iloc[i]

## then define vector of all ones
u = np.zeros(num_var) # cost vector
for i in range(num_var):
    u[i] = 1
    
model = gp.Model()

# Placing the variables in a pandas Series object will allow us to use the dot() function
x = pd.Series(model.addVars(num_var,vtype=GRB.BINARY))

model.setObjective(c.dot(x), GRB.MAXIMIZE)

model.addConstr(u.dot(x) <= num_loans)

model.optimize()    
    
# Extracting the optimal solution and optimal value
print('Optimal expected return:',model.ObjVal)
for i in range(num_var):
    if x[i].X > 0: # most variables are zero, so we just print the non-zero variables
        print('Choose loan',i+1)    


Restricted license - for non-production use only - expires 2024-10-28
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (mac64[rosetta2])

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads



GurobiError: Model too large for size-limited license; visit https://www.gurobi.com/free-trial for a full license

### Partial Investments
Now we want to decide which loan to invest in, and how much should the investment be.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

ret_col = 'ret_OPT'
test_pool = two_stage[ret_col]['test data']
num_var = test_pool.shape[0]
num_loans = 10
min_investment = 100

# Define cost vector
c = np.zeros(num_var)
for i in range(num_var):
    c[i] = test_pool['predicted_return_'+ret_col].iloc[i]*test_pool.funded_amnt.iloc[i]

# Define vector of all ones
u = np.zeros(num_var)
for i in range(num_var):
    u[i] = 1
    
model = gp.Model()

# Define the binary variables x and continuous variables y
x = pd.Series(model.addVars(num_var, vtype=GRB.BINARY))
y = pd.Series(model.addVars(num_var, lb=0, ub=test_pool.funded_amnt.values))

# Set the objective function
model.setObjective(c.dot(y), GRB.MAXIMIZE)

# Add constraints
for i in range(num_var):
    model.addConstr(y[i] <= x[i] * test_pool.funded_amnt.iloc[i])
    model.addConstr(y[i] >= min_investment * x[i])

# Update the budget constraint to be based on the sum of y
model.addConstr(u.dot(y) <= num_loans * test_pool.funded_amnt.mean()) # Adjust the budget constraint as needed

model.optimize()

# Extracting the optimal solution and optimal value
print('Optimal expected return:', model.ObjVal)
for i in range(num_var):
    if x[i].X > 0:
        print('Choose loan', i+1, 'with investment amount:', y[i].X)


### 5.3.2: Maximize profit with budget constraint

In [ ]:
ret_col = 'ret_OPT'
test_pool = two_stage[ret_col]['test data']
num_var = test_pool.shape[0]
num_loans = 100
Budget = 1000000

## first define cost vector
c = np.zeros(num_var) # cost vector
for i in range(num_var):
    c[i] = test_pool['predicted_return_'+ret_col].iloc[i]*test_pool.funded_amnt.iloc[i]

## then define vector of all ones
u = np.zeros(num_var) # cost vector
for i in range(num_var):
    u[i] = 1
    
model = gp.Model()

# Placing the variables in a pandas Series object will allow us to use the dot() function
x = pd.Series(model.addVars(num_var,vtype=GRB.BINARY))
amt = pd.Series(test_pool[0:num_var].loan_amnt)

model.setObjective(c.dot(x), GRB.MAXIMIZE)

model.addConstr(u.dot(x) <= num_loans)
model.addConstr(u.dot(x) >= 0.9*num_loans)
model.addConstr(amt.dot(x) <= Budget)

model.optimize()    
    
# Extracting the optimal solution and optimal value
print('Optimal expected return:',model.ObjVal)
for i in range(num_var):
    if x[i].X > 0: # most variables are zero, so we just print the non-zero variables
        print('Choose loan',i+1)    


In [ ]:
opt_sln_IP2 = np.zeros(num_var)
for i in range(num_var):
        opt_sln_IP2[i] = x[i].X

In [ ]:
sum(opt_sln_IP2)

### Sanity check
Intuitively the optimal solution $x^*$ should sequentially choose the highest return loans. We compare $x^*$ with opt_sln below.

In [ ]:
temp = np.sort(c)
temp = temp[::-1] # in descending order
cutoff = temp[100]
y = np.zeros(num_var)
for i in range(num_var):
    if cutoff<c[i]:
        y[i]=1
print("number of entries that differ:", int((y-opt_sln_IP2).sum()) )

### 5.3.3: Maximize profit with risk-return tradeoff

In [ ]:
## First we need to train a clustering model to estimate the variance of return
n_clusters = 50
train_set = data_dict['train_set']
data_train = data.loc[train_set,:]

# Create a dataframe for testing, including the score
data_test = two_stage[ret_col]['test data']

kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(data_dict['X_train'])
data_train['clusID'] = kmeans.predict(data_dict['X_train'])
data_test['clusID'] = kmeans.predict(data_dict['X_test'])
data_test['volatility'] = 0

for idx in range(n_clusters):
    std_clus = np.std(data_train[ret_col][data_train.clusID == idx])
    data_test.volatility[data_test.clusID == idx] = std_clus

## Specify the parameters of the optimization model
# beta: penalty factor on the risk
beta = 0.9
#Budget = 10.7*1000000

In [ ]:
ret_col = 'ret_OPT'
test_pool = two_stage[ret_col]['test data']
num_var = test_pool.shape[0]
num_loans = 100
Budget = 1000000

## define objective
c = np.zeros(num_var) # cost vector
for i in range(num_var):
    c[i] = (test_pool['predicted_return_'+  ret_col].iloc[i] -
            beta * test_pool.volatility.iloc[i]) * test_pool.loan_amnt.iloc[i]
    
## then define vector of all ones
u = np.zeros(num_var) # cost vector
for i in range(num_var):
    u[i] = 1
    
model = gp.Model()

# Placing the variables in a pandas Series object will allow us to use the dot() function
x = pd.Series(model.addVars(num_var,vtype=GRB.BINARY))
amt = pd.Series(test_pool[0:num_var].loan_amnt)

model.setObjective(c.dot(x), GRB.MAXIMIZE)

model.addConstr(u.dot(x) <= num_loans)
model.addConstr(u.dot(x) >= 0.9*num_loans)
model.addConstr(amt.dot(x) <= Budget)

model.optimize()    
    
# Extracting the optimal solution and optimal value
print('Optimal expected return:',model.ObjVal)
for i in range(num_var):
    if x[i].X > 0: # most variables are zero, so we just print the non-zero variables
        print('Choose loan',i+1)    

 

In [ ]:
opt_sln_IP3 = np.zeros(num_var)
for i in range(num_var):
        opt_sln_IP3[i] = x[i].X

In [ ]:
sum(opt_sln_IP3)

### 5.3.4 Visualization: violin plots for the expected returns of the above strategies

In [ ]:
test_pool['label'] = ['test_pool']*test_pool.shape[0]

# now we create a df for OPT of IP2
test_pool['chosen_IP2'] = opt_sln_IP2
df2 = test_pool[test_pool.chosen_IP2 == 1].copy() # df containing only rows chosen by IP2
test_pool = test_pool.drop(columns=['chosen_IP2']) # drop this columns in order to append
df2 = df2.drop(columns=['chosen_IP2']) # drop this columns in order to append
df2['label'] = ['IP2']*df2.shape[0]

# create a df for OPT of IP3
test_pool['chosen_IP3'] = opt_sln_IP3
df3 = test_pool[test_pool.chosen_IP3 == 1].copy() # df containing only rows chosen by IP3
test_pool = test_pool.drop(columns=['chosen_IP3']) # drop this columns in order to append
df3 = df3.drop(columns=['chosen_IP3']) # drop this columns in order to append
df3['label'] = ['IP3']*df3.shape[0]

# concatenate the dataframes
df_big = test_pool.append([df2,df3])

In [ ]:
ax = sns.violinplot(y=df_big["predicted_return_ret_PESS"], x=df_big["label"], 
                    data=df_big, order=['test_pool','IP2','IP3'])
ax.set_title("Ditribution of expected return of the selected loans")
ax.set_xlabel("Subset of loans")
ax.set_ylabel("ret_PESS")

## 5.4. Sensitivity analysis of the optimization solution by varying the budgets and the number of loans invested in
Build a trade-off curve between the beta value and the optimization objective (expected return). For this we loop through various values of beta from 0.1 through some large number in steps of 0.1 say. Then we create a plot of different pairs of means and stdevns of the expected return of the portfolios as you vary beta. The two axes will then be mean (return) and stdevn (risk).

In [ ]:
num_var = test_pool.shape[0]
Budget = 1000000
beta_min = 0.1
step = 0.1
num_beta = 10
u = np.zeros(num_var) # units vector

for i in range(num_var):
    u[i] = 1

amt = pd.Series(test_pool[0:num_var].loan_amnt)

table = np.zeros([num_beta,2]) # store the return and risk
for j in range(num_beta):
    beta = beta_min + j*step
    
    c = np.zeros(num_var) # cost vector
    for i in range(num_var):
        c[i] = (test_pool['predicted_return_'+  ret_col].iloc[i] - beta * test_pool.volatility.iloc[i]) * test_pool.loan_amnt.iloc[i]
    
    model = gp.Model()

    x = pd.Series(model.addVars(num_var,vtype=GRB.BINARY))

    model.setObjective(c.dot(x), GRB.MAXIMIZE)
    
    model.addConstr(u.dot(x) <= num_loans)
    model.addConstr(u.dot(x) >= 0.9*num_loans)
    model.addConstr(amt.dot(x) <= Budget)

    model.optimize() 
    
    opt = np.zeros(num_var)
    for k in range(num_var):
        opt[k] = x[k].X
    test_pool['chosen'] = opt
    df = test_pool[test_pool.chosen == 1]
    table[j,0] = df['predicted_return_'+  ret_col].sum()
    table[j,1] = df.volatility.sum()

### Visualization: Return vs Risk curve

In [ ]:
plt.plot(table[:,0],table[:,1])
plt.xlabel("return")
plt.ylabel("risk")